# T18b — thử InfoXLM-large ở learning rate thấp hơn

Ở lượt chạy 27/08/2026, **cả ba seed của InfoXLM-large đều không học được**: loss đứng ở
`ln(3) = 1,0986` suốt epoch đầu và bị cơ chế dừng sớm loại.

```
    seed 42: 0.2371   ← KHÔNG HỌC ĐƯỢC, loss đứng ở ln(3), đã loại
    seed 43: 0.1786   ← KHÔNG HỌC ĐƯỢC, loss đứng ở ln(3), đã loại
    seed 44: 0.1670   ← KHÔNG HỌC ĐƯỢC, loss đứng ở ln(3), đã loại
```

Cùng kích thước, cùng độ dài, cùng learning rate `1e-5` với XLM-R-large — vốn chạy tốt 3/3
seed và đạt 0,771. Nên đây là bất ổn riêng của checkpoint InfoXLM, không phải của cấu hình.

## Notebook này thử gì

Hai learning rate thấp hơn, **một seed một epoch mỗi cái**, khoảng 10 phút mỗi lần:

| Lần | lr | Giả thuyết |
|---|---|---|
| A | `5e-6` | Bằng một nửa mức đã hỏng. Đây là cách chữa tiêu chuẩn cho bất ổn của họ RoBERTa cỡ large |
| B | `2e-6` | Nếu A vẫn hỏng thì thử thấp hơn nữa trước khi kết luận |

**Tổng khoảng 25 phút** kể cả cài đặt và tải mô hình 2,24 GB. Rẻ hơn nhiều so với 90 phút của
một lượt ba seed ba epoch.

## Vì sao hạ learning rate là hướng đúng

Loss của InfoXLM không nằm im tuyệt đối mà **nảy quanh** `ln(3)`, từ 1,0131 tới 1,1923. Đó là
dáng của một mô hình đã rơi vào nghiệm thoái hóa — đoán theo tỷ lệ nền của ba lớp — chứ không
phải dáng của một mô hình không được cập nhật gì. Bước cập nhật quá lớn ngay từ đầu là nguyên
nhân hợp lý nhất, và giảm bước là cách chữa trực tiếp.

## Notebook settings

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens`

## Chuẩn bị

Ba ô dưới đây giống hệt notebook T18 chính. Khoảng 2 phút.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 563f74a T18: notebook thử InfoXLM ở learning rate thấp hơn (#41)


In [2]:
# Ô 2 — cài đặt.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate pyvi

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 105.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 99.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 

In [3]:
# Ô 3 — chuẩn bị dữ liệu. Chỉ cần bộ vihallu nên dùng --only. Khoảng 1 phút, chạy CPU.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, ba mô hình đều mở nên vẫn chạy được")

get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py --only vihallu")
get_ipython().system(
    "python -m pytest tests/test_encoder.py tests/test_splits.py"
    " tests/test_metrics.py -q"
)

HF_TOKEN: không có, ba mô hình đều mở nên vẫn chạy được

MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 563f74a T18: notebook thử InfoXLM ở learning rate thấp hơn (#41)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : chưa cài
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/unicorn1209/viha

## Hai lần thử

Chạy nối nhau trong một ô. Lần A hỏng thì script trả mã lỗi nhưng `!` **không** dừng ô, nên
lần B vẫn chạy — đó là điều mong muốn ở đây.

Đọc kết quả: nhìn dòng `macro-F1 từng seed`. Có dấu `← KHÔNG HỌC ĐƯỢC` là hỏng; không có dấu
và điểm quanh 0,70 trở lên là được.

In [4]:
# Ô 4 — hai lần thử, mỗi lần một seed một epoch. Khoảng 20 phút cả hai.
# Mô hình 2,24 GB chỉ tải một lần, lần B dùng lại bản đã lưu đệm.
print("=" * 80, "LẦN A — learning rate 5e-6", "=" * 80, sep=chr(10))
get_ipython().system(
    "python scripts/train_encoder_baseline.py --model infoxlm --seeds 1 --epochs 1 --lr 5e-6"
)

print("=" * 80, "LẦN B — learning rate 2e-6", "=" * 80, sep=chr(10))
get_ipython().system(
    "python scripts/train_encoder_baseline.py --model infoxlm --seeds 1 --epochs 1 --lr 2e-6"
)

LẦN A — learning rate 5e-6

T18 / E09 — TINH CHỈNH INFOXLM
  mô hình               : microsoft/infoxlm-large
  độ dài tối đa         : 512 token
  tách từ tiếng Việt    : không
  batch / epoch / lr    : 8 / 1 / 5e-06
  số seed               : 1
  thiết bị              : cuda
  train / test          : 5,600 / 700 mẫu
config.json: 100%|█████████████████████████████| 513/513 [00:00<00:00, 2.48MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 11.6MB/s]
tokenizer.json: 9.10MB [00:00, 87.8MB/s]
  mẫu bị cắt vì quá dài : 2.5 %
  nhiệt độ trước khi chạy : 42 °C, xung SM 300/1590 MHz (lúc rảnh, không dùng để xét hạ xung)

  --- seed 42 (1/1) ---
pytorch_model.bin: 100%|████████████████████| 2.24G/2.24G [00:09<00:00, 228MB/s]
model.safetensors:   0%|                   | 8.32M/2.24G [00:01<03:19, 11.2MB/s]
Loading weights: 100%|██████████████████████| 389/389 [00:00<00:00, 9597.95it/s]
model.safetensors: 100%|████████████████████| 2.24G/2.24G [00:16<00:00, 138MB/s]
    

## Sau đó làm gì

**Nếu một trong hai lần học được:** chạy đủ ba seed ba epoch ở learning rate đó, khoảng 90 phút.

```
!python scripts/train_encoder_baseline.py --model infoxlm --lr <lr thành công>
```

Rồi sửa `MODELS["infoxlm"]["lr"]` trong `src/vihallulens/detect/encoder.py` cho khớp, để lần
sau không phải nhớ truyền cờ.

**Nếu cả hai đều hỏng:** dừng ở đây, không thử tiếp. Hai mô hình đã đủ để mốc so sánh đứng
vững — XLM-R đạt 0,771 và đó mới là mốc mà đề tài phải vượt. Việc InfoXLM không tinh chỉnh được
tự nó là **một kết quả về độ ổn định** đáng ghi vào báo cáo, không phải một ô trống cần lấp:
nó cho thấy hướng bộ mã hóa đòi dò tham số cho từng checkpoint, còn phương pháp chú ý nội tại
không tinh chỉnh gì nên không có rủi ro này. Đây là luận điểm cho câu hỏi CH2.

Đừng thử quá hai lần. Quota GPU 30 giờ/tuần còn phải để cho T19 và T20.